# Griffschrift: Kopf-Erkenner nachlernen (YOLO)

Dieses Notizbuch erzeugt mit dem Griffschrift-Wandler Trainingsbilder mit bekannter Lösung (runde Köpfe voll/hohl, Kreuz, Kreuz im Kreis, Dreieck voll/hohl, vorgesetztes Doppelkreuz nach Rosenzopf), verschlechtert sie wie Scans, Kopien und Handyfotos und lernt damit einen Erkenner. Am Ende wird `griffschrift_koepfe.zip` heruntergeladen (Modell als ONNX und PyTorch, Lernkurven).

**Vorher:** oben im Menü *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen. Dann *Laufzeit → Alle ausführen*. Das Ganze dauert etwa zwei Stunden; das Fenster muss dabei offen bleiben.

Die Notenblätter der Testpersonen werden hier nicht verwendet, nur vom Wandler gesetzte Stücke (Sätze von Franz Fuchs und eigene Beispiele aus dem Repository).

In [ ]:
#@title 1. Programme einrichten (ca. 5 Minuten)
!git clone -q --depth 1 https://github.com/jpascher/Harmonikabau.git
%cd /content
!pip -q install ultralytics opencv-python-headless
!npm -s i playwright@1 > /dev/null
!npx -y playwright install --with-deps chromium > /dev/null
import subprocess, torch
print('Grafikkarte:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KEINE – bitte T4-GPU einstellen')

In [ ]:
#@title 2. Trainingsbilder mit dem Wandler setzen (ca. 30–40 Minuten)
ANZAHL_STUECKE = 60  #@param {type:"integer"}
import subprocess, time, os
srv = subprocess.Popen(['python3','-m','http.server','8798'], cwd='/content/Harmonikabau/griffschrift-wandler', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
GEN='/content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_erzeuge_wandlerbilder.js'
os.makedirs('/content/raw', exist_ok=True)
# zwei Läufe mit verschiedenen Startwerten, damit andere Stücke und Schreibweisen drankommen
for seed in (11, 23):
    !cd /content && NODE_PATH=/content/node_modules node {GEN} /content/raw {seed} {ANZAHL_STUECKE//2}
!ls /content/raw | wc -l

In [ ]:
#@title 3. Seiten zusammenstellen und verschlechtern (ca. 10 Minuten)
SEITEN = 1500  #@param {type:"integer"}
!python3 /content/Harmonikabau/griffschrift-wandler/omr-space/griffschrift-leser/yolo_daten.py /content/raw /content/ds {SEITEN}

In [ ]:
#@title 4. Lernen (ca. 60–90 Minuten auf einer T4)
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(data='/content/ds/data.yaml', imgsz=960, epochs=60, batch=16, patience=15, project='/content/lauf', name='koepfe', exist_ok=True,
            degrees=0, fliplr=0, flipud=0, mosaic=0.5, hsv_h=0, hsv_s=0, hsv_v=0.3, scale=0.25, translate=0.05)
metrics = model.val()
print(metrics.box.map50, metrics.box.map)

In [ ]:
#@title 5. Als ONNX speichern und herunterladen
best = YOLO('/content/lauf/koepfe/weights/best.pt')
best.export(format='onnx', imgsz=960, opset=12, simplify=True)
!cd /content/lauf/koepfe && cp weights/best.pt griffschrift_koepfe.pt && cp weights/best.onnx griffschrift_koepfe.onnx && zip -q -r /content/griffschrift_koepfe.zip griffschrift_koepfe.pt griffschrift_koepfe.onnx results.csv results.png confusion_matrix.png args.yaml
from google.colab import files
files.download('/content/griffschrift_koepfe.zip')